# MMIDAS Pipeline Output Validation

Sanity-checks the outputs of `MMIDAS_DataPrep` -> `MMIDAS_Train` -> `MMIDAS_Analyze`.

Fill in the `gs://` paths in the **Config** cell below with the actual outputs from your run,
then run all cells top to bottom.

Each check prints one of:
- `[PASS]` / `[FAIL]` - an automated, threshold-based check
- `[REVIEW]` - something (usually a figure) that needs a human eyeball rather than a hard threshold

Requires `gsutil` on PATH and authenticated (`gcloud auth login` / `gcloud auth application-default login`),
plus the packages imported below (`anndata`, `matplotlib`, `numpy`).

In [ ]:
import io
import json
import os
import pickle
import subprocess
import tarfile

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np

## Config

Fill in the actual `gs://` output paths from your run.

In [ ]:
CONFIG = {
    "dataprep": {
        "preprocessed_h5ad": "gs://your-bucket/results/Mouse_ALM-VISp_cpm.h5ad",
        # The actual selected_genes CSV passed to MMIDAS_DataPrep for this run --
        # used to check the retained gene count against the real input rather
        # than a hardcoded guess (some symbols may be missing from genes_rows
        # and get dropped; 01_data_prep.py logs a WARNING listing which).
        "selected_genes_csv": "gs://your-bucket/genes_SS_ALM-VISp.csv",
    },

    "train": {
        "evaluation_results_json": "gs://your-bucket/results/evaluation_results.json",
        "checkpoints_manifest":    "gs://your-bucket/results/checkpoints_manifest.json",
        "model_tar":               "gs://your-bucket/results/model.tar.gz",
        # Array[File] output of MMIDAS_Train.evaluation_figures. Either paste the
        # explicit list of gs:// URIs (from Terra's data table / Cromwell metadata),
        # or give a single gs:// prefix and we'll glob it for *.png.
        "evaluation_figures": "gs://your-bucket/cromwell-execution/MMIDAS_Train/.../call-Evaluate/",
    },

    "analyze": {
        "clusterability_manifest":  "gs://your-bucket/results/clusterability_manifest.json",
        "clusterability_figures":   "gs://your-bucket/cromwell-execution/MMIDAS_Analyze/.../call-Clusterability/",
        "state_traversal_manifest": "gs://your-bucket/results/state_traversal_manifest.json",
        "state_traversal_figures":  "gs://your-bucket/cromwell-execution/MMIDAS_Analyze/.../call-StateTraversal/",
    },

    # Optional. classify_manifest.json / clustering_tar are intermediate outputs of
    # the Classify (03b) task -- NOT part of MMIDAS_Analyze's final workflow outputs.
    # If you pull them out of the Cromwell execution directory, this notebook can
    # compute real accuracy/silhouette numbers instead of just eyeballing the
    # classAcc_RF / SC_K_* figures.
    "classify_optional": {
        "classify_manifest": None,  # e.g. "gs://your-bucket/.../call-Classify/classify_manifest.json"
        "clustering_tar":    None,  # e.g. "gs://your-bucket/.../call-Classify/clustering.tar.gz"
    },

    # Expected values -- should match the inputs used for this run.
    "expected": {
        "neuronal_classes":  ["GABAergic", "Glutamatergic"],
        "remove_clusters":   ["Low Quality", "CR Lhx5", "Meis2 Adamts19"],
        "n_categories":      120,     # MMIDAS_Train.n_categories input (pruning ceiling)
        "k_select_thr":      0.95,    # MMIDAS_Train.k_select_thr input
    },
}

SCRATCH_DIR = "/tmp/mmidas_validation"

## Helpers

In [ ]:
os.makedirs(SCRATCH_DIR, exist_ok=True)

CHECKS = []
REVIEW_ITEMS = []


def check(name, passed, detail=""):
    """Record an automated pass/fail check."""
    CHECKS.append({"name": name, "passed": bool(passed), "detail": detail})
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {name}" + (f" -- {detail}" if detail else ""))


def review(name, detail=""):
    """Flag something that needs a human eyeball rather than a hard threshold."""
    REVIEW_ITEMS.append({"name": name, "detail": detail})
    print(f"[REVIEW] {name}" + (f" -- {detail}" if detail else ""))


def gcs_download(gs_path, dest_dir=SCRATCH_DIR):
    """Download a single gs:// object and return its local path."""
    os.makedirs(dest_dir, exist_ok=True)
    local_path = os.path.join(dest_dir, os.path.basename(gs_path))
    if not os.path.exists(local_path):
        subprocess.run(["gsutil", "-q", "cp", gs_path, local_path], check=True)
    return local_path


def gcs_list(prefix, pattern="*.png"):
    """List objects under a gs:// prefix (recursive) matching a glob pattern."""
    result = subprocess.run(
        ["gsutil", "ls", os.path.join(prefix.rstrip("/"), "**", pattern)],
        capture_output=True, text=True,
    )
    return [line for line in result.stdout.splitlines() if line.strip()]


def resolve_paths(value, pattern="*.png"):
    """CONFIG figure entries may be a list of gs:// URIs or a single prefix to glob."""
    if value is None:
        return []
    if isinstance(value, (list, tuple)):
        return list(value)
    return gcs_list(value, pattern=pattern)


def load_json_gcs(gs_path):
    with open(gcs_download(gs_path)) as fh:
        return json.load(fh)


def load_h5ad_gcs(gs_path):
    return ad.read_h5ad(gcs_download(gs_path))


def extract_tar_gcs(gs_path, subdir):
    local = gcs_download(gs_path)
    dest = os.path.join(SCRATCH_DIR, subdir)
    os.makedirs(dest, exist_ok=True)
    with tarfile.open(local) as tf:
        tf.extractall(dest)
    return dest


def show_image(gs_or_local_path, title=None, figsize=(6, 6)):
    path = gcs_download(gs_or_local_path) if str(gs_or_local_path).startswith("gs://") else gs_or_local_path
    img = plt.imread(path)
    plt.figure(figsize=figsize)
    plt.imshow(img)
    plt.axis("off")
    if title:
        plt.title(title, fontsize=10)
    plt.show()

## Stage 1 -- DataPrep (`preprocessed_h5ad`)

Checks: gene count matches the selected gene list, no NaN/Inf/negative values (data is log-CPM),
cluster count is plausible, excluded clusters are actually absent, and `class` only contains the
configured neuronal classes.

In [ ]:
adata = load_h5ad_gcs(CONFIG["dataprep"]["preprocessed_h5ad"])
print(adata)

n_cells, n_genes = adata.shape
print(f"\n{n_cells} cells x {n_genes} genes")
print(f"obs columns: {list(adata.obs.columns)}")

selected_genes_path = CONFIG["dataprep"].get("selected_genes_csv")
if selected_genes_path:
    import pandas as pd
    n_selected = len(pd.read_csv(gcs_download(selected_genes_path)))
    print(f"\n{n_selected} genes in selected_genes CSV")
    # 01_data_prep.py drops any selected_genes symbol missing from genes_rows and
    # logs a WARNING listing which, so n_genes may be slightly below n_selected --
    # but it can never exceed it, and a large deficit signals a real problem.
    check(
        "gene count matches selected_genes CSV (allowing for symbols missing from genes_rows)",
        n_genes <= n_selected and n_genes >= 0.9 * n_selected,
        f"got {n_genes} retained vs {n_selected} in selected_genes CSV",
    )
else:
    review(
        "gene count vs selected_genes CSV",
        f"got {n_genes} genes -- set CONFIG['dataprep']['selected_genes_csv'] to the actual "
        "input file for this run to auto-check this instead of eyeballing it",
    )

X = adata.X.toarray() if not isinstance(adata.X, np.ndarray) else adata.X
check("no NaN/Inf values in X", bool(np.isfinite(X).all()))
check("no negative values in X (log-CPM)", bool(X.min() >= 0), f"min={X.min():.4f}")

if "cluster" in adata.obs:
    n_clusters = adata.obs["cluster"].nunique()
    print(f"\n{n_clusters} unique clusters")
    check("cluster count in a plausible range (5-200)", 5 <= n_clusters <= 200, f"got {n_clusters}")

    removed = set(CONFIG["expected"]["remove_clusters"]) & set(adata.obs["cluster"].unique())
    check("excluded clusters are absent from output", len(removed) == 0, f"found: {removed}" if removed else "")

if "class" in adata.obs:
    classes = set(adata.obs["class"].unique())
    expected_classes = set(CONFIG["expected"]["neuronal_classes"])
    check(
        "obs['class'] only contains expected neuronal classes",
        classes <= expected_classes,
        f"found: {classes}",
    )
    print("\nCells per class:")
    print(adata.obs["class"].value_counts())

# Stash for cross-stage consistency checks below.
DATAPREP_N_GENES = n_genes

## Stage 2 -- Train (`evaluation_results.json`, `checkpoints_manifest.json`, evaluation figures)

Checks: gene count matches DataPrep's output, `model_order` is below the `n_categories` pruning
ceiling (if it's equal to the ceiling, pruning likely didn't converge), and `avg_consensus` meets
`k_select_thr`. Figures are shown inline for manual review.

In [ ]:
eval_results = load_json_gcs(CONFIG["train"]["evaluation_results_json"])
print(json.dumps(eval_results, indent=2))

model_order   = eval_results["model_order"]
n_categories  = eval_results["n_categories"]
avg_consensus = eval_results["avg_consensus"]
k_select_thr  = eval_results["k_select_thr"]
n_gene        = eval_results["n_gene"]

check(
    "n_gene in evaluation_results matches DataPrep gene count",
    n_gene == DATAPREP_N_GENES,
    f"train n_gene={n_gene}, dataprep n_genes={DATAPREP_N_GENES}",
)

check(
    "model_order is below the n_categories pruning ceiling",
    model_order < n_categories,
    f"model_order={model_order}, n_categories={n_categories}"
    + ("  <-- pruning may not have converged" if model_order >= n_categories else ""),
)

check("model_order is at least 2", model_order >= 2, f"got {model_order}")

check(
    "avg_consensus meets k_select_thr",
    avg_consensus >= k_select_thr,
    f"avg_consensus={avg_consensus:.4f}, k_select_thr={k_select_thr}",
)

In [ ]:
ckpt_manifest = load_json_gcs(CONFIG["train"]["checkpoints_manifest"])

check(
    "checkpoints_manifest n_categories matches evaluation_results",
    ckpt_manifest.get("n_categories") == n_categories,
    f"ckpt={ckpt_manifest.get('n_categories')}, eval={n_categories}",
)
print(f"checkpoints in manifest: {len(ckpt_manifest.get('checkpoints', []))}")

In [ ]:
eval_figure_paths = resolve_paths(CONFIG["train"]["evaluation_figures"], pattern="*.png")
print(f"Found {len(eval_figure_paths)} evaluation figures")

for p in eval_figure_paths:
    name = os.path.basename(p)
    if name.startswith("consensus_T1_vs_T2"):
        review("consensus bubble plot", "expect a strong diagonal -- arms should mostly agree on assignment")
    elif name.startswith("norm_consensus"):
        review("normalized consensus plot", f"average consensus should be >= {k_select_thr}")
    elif name.startswith("state_mu"):
        review("state-space scatter", "expect visually separated clusters, not one undifferentiated blob")
    show_image(p, title=name)

## Stage 3 -- Analyze: Clusterability (`clusterability_manifest.json` + figures)

Checks `model_order` is consistent with Train's `evaluation_results.json`, then shows the
classification-accuracy bar chart, silhouette curve, and confusion-matrix heatmaps for review.

In [ ]:
clust_manifest = load_json_gcs(CONFIG["analyze"]["clusterability_manifest"])
print(json.dumps(clust_manifest, indent=2))

check(
    "clusterability model_order matches Train's evaluation_results",
    clust_manifest["model_order"] == model_order,
    f"clusterability={clust_manifest['model_order']}, train={model_order}",
)

In [ ]:
clust_figure_paths = resolve_paths(CONFIG["analyze"]["clusterability_figures"], pattern="*.png")
print(f"Found {len(clust_figure_paths)} clusterability figures")

for p in clust_figure_paths:
    name = os.path.basename(p)
    if name.startswith("classAcc_RF"):
        review(
            "RF classification accuracy bar chart",
            "MMIDAS lowD embedding should be comparable to or better than the PCA baseline",
        )
    elif name.startswith("SC_K_"):
        review("silhouette score curve", "most categories should have positive silhouette scores")
    elif name.startswith("conf_"):
        review("confusion matrix heatmap", f"expect a block-diagonal pattern ({name})")
    show_image(p, title=name)

### Optional: numeric accuracy/silhouette check

Only runs if you supplied `classify_manifest` / `clustering_tar` in the config above (these are
intermediate outputs of the 03b Classify task, not part of `MMIDAS_Analyze`'s final outputs).
Computes actual mean RF accuracy and the fraction of categories with positive silhouette score
per pickle, instead of relying purely on the figures above.

In [ ]:
classify_cfg = CONFIG["classify_optional"]

if classify_cfg["classify_manifest"] and classify_cfg["clustering_tar"]:
    classify_manifest = load_json_gcs(classify_cfg["classify_manifest"])
    clustering_root = extract_tar_gcs(classify_cfg["clustering_tar"], "clustering")

    for pickle_path in classify_manifest["pickles"]:
        local_pickle = os.path.join(clustering_root, "clustering", os.path.basename(pickle_path))
        if not os.path.exists(local_pickle):
            print(f"  (skipping, not found locally: {os.path.basename(pickle_path)})")
            continue
        with open(local_pickle, "rb") as fh:
            data = pickle.load(fh)
        acc = data["acc_T_adj"]
        sc_flat = np.concatenate([np.atleast_1d(s) for s in data["sc_T"]])
        name = os.path.basename(pickle_path)
        print(
            f"{name}: mean acc={acc.mean():.3f} (+/-{acc.std():.3f}), "
            f"pct categories w/ positive silhouette={100 * (sc_flat > 0).mean():.1f}%"
        )
else:
    print(
        "classify_manifest / clustering_tar not provided -- skipping numeric accuracy/silhouette "
        "checks. Relying on the figures above for a visual review instead."
    )

## Stage 3 -- Analyze: State Traversal (`state_traversal_manifest.json` + figures)

Checks `model_order` consistency and that the traversal ran for the expected number of
categories, then shows a sample of the traversal figures for review.

In [ ]:
state_manifest = load_json_gcs(CONFIG["analyze"]["state_traversal_manifest"])
print(json.dumps(state_manifest, indent=2))

check(
    "state_traversal model_order matches Train's evaluation_results",
    state_manifest["model_order"] == model_order,
    f"state_traversal={state_manifest['model_order']}, train={model_order}",
)

n_selected_cats = state_manifest["n_selected_cats"]
selected_c = state_manifest["selected_c"]
check(
    "state_traversal ran for the requested number of categories",
    len(selected_c) == n_selected_cats or n_selected_cats == 0,
    f"selected_c has {len(selected_c)} entries, n_selected_cats={n_selected_cats}",
)

In [ ]:
state_figure_paths = resolve_paths(CONFIG["analyze"]["state_traversal_figures"], pattern="*.png")
print(f"Found {len(state_figure_paths)} state traversal figures")

# These can be numerous -- show a handful rather than all of them.
N_TO_SHOW = 8
for p in state_figure_paths[:N_TO_SHOW]:
    review(
        "state traversal figure",
        "gene expression / pathway score should trend smoothly along the traversal path, not look like noise",
    )
    show_image(p, title=os.path.basename(p))

if len(state_figure_paths) > N_TO_SHOW:
    print(f"... and {len(state_figure_paths) - N_TO_SHOW} more not shown (raise N_TO_SHOW to see more)")

## Summary

In [ ]:
n_pass = sum(c["passed"] for c in CHECKS)
n_fail = len(CHECKS) - n_pass

print("=" * 60)
print(f"Automated checks: {n_pass}/{len(CHECKS)} passed")
if n_fail:
    print("\nFAILED:")
    for c in CHECKS:
        if not c["passed"]:
            print(f"  - {c['name']}: {c['detail']}")

print(f"\n{len(REVIEW_ITEMS)} figures/items flagged for manual review (see inline images above).")